# Finly — Fase 2: Categorização com IA (Ollama + Llama 3.1)

Neste notebook vamos:
1. Carregar o extrato simulado
2. Usar o Llama 3.1 **rodando localmente** via Ollama para categorizar cada transação
3. Avaliar a qualidade comparando com o gabarito

> **Diferencial de portfólio:** rodar um LLM local mostra que você entende a stack além das APIs pagas — sem custo, sem limite de requisições, sem dependência de internet.

## 1. Imports e configuração

In [ ]:
import pandas as pd
import requests
import json
import time

# Ollama roda localmente na porta 11434 — sem chave de API necessária!
OLLAMA_URL = "http://localhost:11434/api/generate"
MODELO     = "llama3.2"

# Testa se o Ollama está rodando
try:
    r = requests.get("http://localhost:11434", timeout=3)
    print("✅ Ollama está rodando!")
except Exception:
    print("❌ Ollama não está rodando. Abra o app Ollama e tente novamente.")

✅ Ollama está rodando!


## 2. Carregar o extrato

In [27]:
extrato = pd.read_csv("../extrato_simulado.csv", parse_dates=["data"])

print(f"Total de transações: {len(extrato)}")
print(f"Período: {extrato['data'].min().date()} → {extrato['data'].max().date()}")
print()
extrato.head(8)

Total de transações: 459
Período: 2024-01-01 → 2024-12-26



,data,descricao,valor
0,2024-01-01,Hortifruti Verde,419.61
1,2024-01-01,iFood - McDonald's,70.35
2,2024-01-01,iFood - Frango Assado,45.73
3,2024-01-02,Boate Floresta,42.98
4,2024-01-03,iFood - Frango Assado,50.32
5,2024-01-04,Posto Shell,144.26
6,2024-01-06,Amazon Livros,87.87
7,2024-01-07,Magazine Luiza,360.77


## 3. Função de categorização com Llama 3.1

O Ollama expõe uma API REST local — chamamos ela com `requests`, igual a qualquer API web.
Processamos **10 transações por lote** (menor que com o Gemini porque o modelo local é mais lento).

In [28]:
CATEGORIAS_VALIDAS = [
    "alimentação", "delivery", "transporte", "lazer",
    "saúde", "vestuário", "educação", "casa", "outros"
]

def categorizar_lote(descricoes: list[str]) -> list[str]:
    """
    Envia um lote de descrições pro Llama 3.1 via Ollama
    e retorna a lista de categorias.
    """
    lista = "\n".join([f"{i+1}. {d}" for i, d in enumerate(descricoes)])

    prompt = f"""Você é um sistema de categorização de gastos pessoais brasileiro.
Categorize cada transação abaixo usando EXATAMENTE uma das categorias: {', '.join(CATEGORIAS_VALIDAS)}

Transações:
{lista}

Responda APENAS com um JSON válido, sem texto adicional, sem explicações:
{{"categorias": ["cat1", "cat2", ...]}}

A lista deve ter exatamente {len(descricoes)} itens na mesma ordem."""

    payload = {
        "model": MODELO,
        "prompt": prompt,
        "stream": False,        # recebe resposta completa de uma vez
        "format": "json",       # força o modelo a responder em JSON
        "options": {
            "temperature": 0,   # zero = respostas mais determinísticas
            "num_predict": 200  # limita tokens da resposta
        }
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()

    texto = response.json()["response"].strip()
    # Remove eventuais blocos de código markdown
    texto = texto.replace("```json", "").replace("```", "").strip()

    resultado = json.loads(texto)
    cats = resultado["categorias"]

    # Garante que categorias inválidas viram 'outros'
    cats = [c if c in CATEGORIAS_VALIDAS else "outros" for c in cats]

    # Garante que o tamanho bate com o lote
    while len(cats) < len(descricoes):
        cats.append("outros")

    return cats[:len(descricoes)]


# Teste rápido — deve retornar: delivery, transporte, saúde
teste = ["iFood - McDonald's", "Posto Shell", "Farmácia Araújo"]
print("Teste rápido (aguarde ~10s):")
resultado_teste = categorizar_lote(teste)
for desc, cat in zip(teste, resultado_teste):
    print(f"  {desc:30s} → {cat}")

Teste rápido (aguarde ~10s):
  iFood - McDonald's             → alimentação
  Posto Shell                    → transporte
  Farmácia Araújo                → saúde


## 4. Processar todas as transações

> ⏱️ Isso vai levar alguns minutos — o Llama roda na sua CPU/GPU local. Aguarde.

In [29]:
TAMANHO_LOTE = 10  # menor que com APIs pagas — modelo local é mais lento

descricoes    = extrato["descricao"].tolist()
categorias_ia = []
total_lotes   = (len(descricoes) + TAMANHO_LOTE - 1) // TAMANHO_LOTE
inicio        = time.time()

for i in range(0, len(descricoes), TAMANHO_LOTE):
    lote      = descricoes[i : i + TAMANHO_LOTE]
    num_lote  = i // TAMANHO_LOTE + 1

    try:
        cats = categorizar_lote(lote)
        categorias_ia.extend(cats)

        elapsed  = time.time() - inicio
        restante = (elapsed / num_lote) * (total_lotes - num_lote)
        print(f"Lote {num_lote:02d}/{total_lotes} ✅  "
              f"({len(cats)} transações)  "
              f"~{int(restante)}s restantes")

    except Exception as e:
        print(f"Lote {num_lote:02d}/{total_lotes} ❌  Erro: {e}")
        categorias_ia.extend(["outros"] * len(lote))

print(f"\n✅ Concluído em {int(time.time()-inicio)}s — {len(categorias_ia)} transações categorizadas")

Lote 01/46 ✅  (10 transações)  ~1867s restantes
Lote 02/46 ✅  (10 transações)  ~1928s restantes
Lote 03/46 ✅  (10 transações)  ~1820s restantes
Lote 04/46 ✅  (10 transações)  ~1762s restantes
Lote 05/46 ✅  (10 transações)  ~1705s restantes
Lote 06/46 ✅  (10 transações)  ~1694s restantes
Lote 07/46 ✅  (10 transações)  ~1634s restantes
Lote 08/46 ✅  (10 transações)  ~1589s restantes
Lote 09/46 ✅  (10 transações)  ~1548s restantes
Lote 10/46 ✅  (10 transações)  ~1505s restantes
Lote 11/46 ✅  (10 transações)  ~1469s restantes
Lote 12/46 ✅  (10 transações)  ~1429s restantes
Lote 13/46 ✅  (10 transações)  ~1385s restantes
Lote 14/46 ✅  (10 transações)  ~1348s restantes
Lote 15/46 ✅  (10 transações)  ~1303s restantes
Lote 16/46 ✅  (10 transações)  ~1260s restantes
Lote 17/46 ✅  (10 transações)  ~1216s restantes
Lote 18/46 ✅  (10 transações)  ~1175s restantes
Lote 19/46 ✅  (10 transações)  ~1137s restantes
Lote 20/46 ✅  (10 transações)  ~1096s restantes
Lote 21/46 ✅  (10 transações)  ~1052s re

## 5. Avaliar a qualidade da IA

In [30]:
extrato["categoria_ia"] = categorias_ia

# Carrega gabarito
gabarito = pd.read_csv("../dados_completos.csv", parse_dates=["data"])
extrato["categoria_real"] = gabarito["categoria"]

# Acurácia geral
acertos  = (extrato["categoria_ia"] == extrato["categoria_real"]).sum()
total    = len(extrato)
acuracia = acertos / total * 100

print(f"Acurácia geral: {acuracia:.1f}%  ({acertos}/{total} acertos)")
print()

# Acurácia por categoria
print("Acurácia por categoria:")
for cat in sorted(CATEGORIAS_VALIDAS):
    subset = extrato[extrato["categoria_real"] == cat]
    if len(subset) == 0:
        continue
    acc   = (subset["categoria_ia"] == subset["categoria_real"]).mean() * 100
    barra = "█" * int(acc / 5)
    print(f"  {cat:15s} {acc:5.1f}%  {barra}")

print()

# Erros mais comuns — útil pra narrativa do notebook
erros = extrato[extrato["categoria_ia"] != extrato["categoria_real"]]
if len(erros) > 0:
    print("Exemplos de erros:")
    print(erros[["descricao", "categoria_real", "categoria_ia"]].head(10).to_string(index=False))

Acurácia geral: 78.4%  (360/459 acertos)

Acurácia por categoria:
  alimentação      92.6%  ██████████████████
  casa              8.3%  █
  delivery         67.3%  █████████████
  educação        100.0%  ████████████████████
  lazer            89.6%  █████████████████
  saúde           100.0%  ████████████████████
  transporte       81.5%  ████████████████
  vestuário        87.5%  █████████████████

Exemplos de erros:
            descricao categoria_real categoria_ia
       Magazine Luiza           casa       outros
  Rappi - Burger King       delivery    vestuário
         Hering Store      vestuário  alimentação
          Posto Shell     transporte       outros
  Rappi - Açaí Brasil       delivery        lazer
iFood - Frango Assado       delivery  alimentação
          Posto Shell     transporte       outros
  Rappi - Açaí Brasil       delivery        lazer
   iFood - McDonald's       delivery  alimentação
  Rappi - Açaí Brasil       delivery        lazer


## 6. Salvar resultado

In [31]:
extrato_categorizado = extrato[["data", "descricao", "valor", "categoria_ia"]].copy()
extrato_categorizado.rename(columns={"categoria_ia": "categoria"}, inplace=True)
extrato_categorizado.to_csv("../extrato_categorizado.csv", index=False)

print("✅ Salvo em extrato_categorizado.csv")
print()
print("Distribuição final das categorias:")
print(extrato_categorizado["categoria"].value_counts().to_string())

✅ Salvo em extrato_categorizado.csv

Distribuição final das categorias:
categoria
transporte     128
alimentação    111
delivery       103
lazer           55
outros          19
vestuário       15
saúde           13
educação        11
casa             4
